# Laboratório 3 — Busca em Espaço de Estados: BFS vs A*

**Disciplina:** CC7711 — Inteligência Artificial e Robótica  
**Data:** 25/02/2026  

---

## Objetivo

Para entender os mecanismos de busca, resolveremos um problema clássico.
- Espaço de Estados: Uma grade 3x3 com 8 peças numeradas e um espaço vazio.

Objetivo: Ordenar as peças de 1 a 8 movendo o espaço vazio.
Desafio: Encontrar a sequência de movimentos que leva do estado inicial bagunçado ao estado ordenado


---

## Passo 1 — Modelagem: Classe `PuzzleState`

Para aplicar qualquer algoritmo de busca, primeiro é preciso modelar o problema como um **espaço de estados**. A classe `PuzzleState` encapsula:

| Atributo | Descrição |
|----------|-----------|
| `board` | Tupla imutável de 9 inteiros representando o tabuleiro linearizado (linha a linha) |
| `parent` | Referência ao estado pai (permite reconstruir o caminho) |
| `action` | Ação que gerou este estado a partir do pai (`'Cima'`, `'Baixo'`, etc.) |
| `cost` | Custo acumulado `g(n)` — profundidade no grafo de busca |

O método central é `get_successors()`:

1. Localiza o índice do espaço vazio (`0`) na tupla.
2. Converte o índice linear em coordenadas `(linha, coluna)` via `divmod(idx, 3)`.
3. Para cada uma das 4 direções, verifica se o movimento permanece dentro do tabuleiro (`0 ≤ nova_linha < 3` e `0 ≤ nova_coluna < 3`).
4. Gera uma nova tupla com as posições do `0` e do vizinho trocadas, criando um novo `PuzzleState` filho.

O método `__lt__` foi adicionado para que o `heapq` do Python possa desempatar elementos com mesmo custo `f` sem lançar `TypeError`.

In [9]:
from collections import deque
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, action=None, cost=0):
        self.board = board  # Tupla, ex: (1, 2, 3, 4, 0, 5, 6, 7, 8)
        self.parent = parent
        self.action = action
        self.cost = cost

    def is_goal(self):
        return self.board == (1, 2, 3, 4, 5, 6, 7, 8, 0)

    def get_successors(self):
        successors = []
        zero_idx = self.board.index(0)
        row, col = divmod(zero_idx, 3)
        moves = {'Cima': (-1, 0), 'Baixo': (1, 0), 'Esquerda': (0, -1), 'Direita': (0, 1)}

        for action, (dr, dc) in moves.items():
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_idx = new_row * 3 + new_col
                new_board = list(self.board)
                new_board[zero_idx], new_board[new_idx] = new_board[new_idx], new_board[zero_idx]
                successors.append(
                    PuzzleState(tuple(new_board), parent=self, action=action, cost=self.cost + 1)
                )
        return successors

    def __lt__(self, other):
        # Necessário para heapq quando os custos f são iguais
        return self.cost < other.cost

print("PuzzleState definido com sucesso.")

PuzzleState definido com sucesso.


---

## Passo 2 — Busca em Largura (BFS)

O BFS é um algoritmo de busca **não-informado** (cego). Ele explora o grafo de estados camada por camada, garantindo encontrar a solução de **menor profundidade** (ótima quando o custo de cada passo é unitário).

**Estrutura de dados:** `deque` (fila de dupla extremidade) do módulo `collections`, que oferece `O(1)` tanto no `append` (inserção no final) quanto no `popleft` (remoção do início) — essencial para não degradar o algoritmo a `O(n)` por operação.

**Estratégia de marcação:** O estado é adicionado ao conjunto `explored` **no momento do enfileiramento**, não no da expansão. Isso evita que o mesmo estado apareça múltiplas vezes na fila e garante que o primeiro caminho encontrado (mais curto) seja o único processado.

**Complexidade:** No pior caso, o BFS visita todos os estados acessíveis. Para o 8-Puzzle isso é até ~181.440 estados (metade dos 362.880), tornando-o inviável para instâncias difíceis sem suporte de memória considerável.

In [10]:
def bfs(start_state):
    """Busca em largura (BFS). Retorna (estado_objetivo, nós_expandidos)."""
    queue = deque([start_state])
    explored = set()
    explored.add(start_state.board)
    nodes_expanded = 0

    while queue:
        state = queue.popleft()
        nodes_expanded += 1

        if state.is_goal():
            return state, nodes_expanded

        for successor in state.get_successors():
            if successor.board not in explored:
                explored.add(successor.board)
                queue.append(successor)

    return None, nodes_expanded  # sem solução

print("bfs definido com sucesso.")

bfs definido com sucesso.


---

## Passo 3 — A\* com Heurística de Manhattan

O **A\*** é um algoritmo de busca **informada**. Ele mantém uma fila de prioridade ordenada pela função de avaliação:

$$f(n) = g(n) + h(n)$$

onde:
- $g(n)$ = custo acumulado do caminho desde o estado inicial até o nó $n$ (profundidade).
- $h(n)$ = estimativa heurística do custo de $n$ até o objetivo.

### Implementação

- Usamos `heapq` do Python (heap mínimo binário) para obter sempre o nó com menor $f$ em $O(\log n)$.
- Estados são marcados como explorados **na expansão** (não no enfileiramento) porque um mesmo estado pode ser redescoberto com custo $g$ menor através de outro caminho — ao chegarmos a ele com custo maior, pulamos com `continue`.

In [11]:
def manhattan_distance(board):
    """Soma das distâncias de Manhattan de cada peça até sua posição objetivo."""
    distance = 0
    for idx, tile in enumerate(board):
        if tile != 0:  # ignora o espaço vazio
            goal_row, goal_col = divmod(tile - 1, 3)   # posição objetivo (0-indexado)
            curr_row, curr_col = divmod(idx, 3)          # posição atual
            distance += abs(goal_row - curr_row) + abs(goal_col - curr_col)
    return distance


def a_star(start_state):
    """A* com heurística de Manhattan. Retorna (estado_objetivo, nós_expandidos)."""
    h0 = manhattan_distance(start_state.board)
    # Fila de prioridade: (f = g + h, estado)
    queue = [(h0, start_state)]
    explored = set()
    nodes_expanded = 0

    while queue:
        f, state = heapq.heappop(queue)

        if state.board in explored:
            continue  # já foi expandido com custo menor

        explored.add(state.board)
        nodes_expanded += 1

        if state.is_goal():
            return state, nodes_expanded

        for successor in state.get_successors():
            if successor.board not in explored:
                g = successor.cost
                h = manhattan_distance(successor.board)
                heapq.heappush(queue, (g + h, successor))

    return None, nodes_expanded  # sem solução

print("manhattan_distance e a_star definidos com sucesso.")

manhattan_distance e a_star definidos com sucesso.


---

## Análise Comparativa

A célula abaixo executa ambos os algoritmos no mesmo estado inicial difícil e imprime:
- Número de nós expandidos por cada algoritmo
- Tamanho (número de passos) da solução encontrada
- O caminho completo de ações
- A razão entre os nós expandidos (o quanto o BFS "desperdiçou" em relação ao A\*)

O estado inicial utilizado para o teste é:

```
5  2  8
4  1  7
_  3  6
```

Este estado está a **22 movimentos** do objetivo — considerado um caso difícil para o 8-Puzzle.

In [12]:
import time

def get_path(state):
    """Reconstrói a sequência de ações do estado inicial até o objetivo."""
    path = []
    while state.parent:
        path.append(state.action)
        state = state.parent
    path.reverse()
    return path

# ──────────────────────────────────────────────
# Estado inicial — altere para testar diferentes dificuldades
# Fácil (~2 passos):   (1, 2, 3, 4, 0, 5, 7, 8, 6)
# Médio (~10 passos):  (1, 2, 3, 4, 5, 6, 0, 7, 8)
# Difícil (~20 passos):(5, 2, 8, 4, 1, 7, 0, 3, 6)
# ──────────────────────────────────────────────
initial_board = (5, 2, 8, 4, 1, 7, 0, 3, 6)
bfsnow = time.time()
start_bfs   = PuzzleState(initial_board)
start_astar = PuzzleState(initial_board)

bfs_time = time.time()
bfs_solution,   bfs_nodes   = bfs(start_bfs)
bfs_time = time.time() - bfs_time

astar_time = time.time()
astar_solution, astar_nodes = a_star(start_astar)
astar_time = time.time() - astar_time

bfs_path   = get_path(bfs_solution)
astar_path = get_path(astar_solution)

print("=" * 40)
print("               BFS")
print("=" * 40)
print(f"  Nós expandidos : {bfs_nodes}")
print(f"  Passos solução : {len(bfs_path)}")
print(f"  Tempo gasto     : {bfs_time:.4f} segundos")

print()
print("=" * 40)
print("               A*")
print("=" * 40)
print(f"  Nós expandidos : {astar_nodes}")
print(f"  Passos solução : {len(astar_path)}")
print(f"  Tempo gasto     : {astar_time:.4f} segundos")

print()
print("=" * 40)
print("           ANÁLISE COMPARATIVA")
print("=" * 40)
ratio = bfs_nodes / astar_nodes
print(f"  BFS expandiu {ratio:.1f}x mais nós que o A*!")
print(f"  Redução alcançada pelo A*: {(1 - astar_nodes/bfs_nodes)*100:.1f}% menos nós.")
print(f"  Tempo BFS: {bfs_time:.4f} s, Tempo A*: {astar_time:.4f} s")

               BFS
  Nós expandidos : 85698
  Passos solução : 22
  Tempo gasto     : 0.5114 segundos

               A*
  Nós expandidos : 672
  Passos solução : 22
  Tempo gasto     : 0.0054 segundos

           ANÁLISE COMPARATIVA
  BFS expandiu 127.5x mais nós que o A*!
  Redução alcançada pelo A*: 99.2% menos nós.
  Tempo BFS: 0.5114 s, Tempo A*: 0.0054 s


---

## Discussão dos Resultados

### Resultados obtidos

Para o estado inicial `(5, 2, 8, 4, 1, 7, 0, 3, 6)` — 22 passos do objetivo:

| Métrica               | BFS       | A\*     | Redução   |
|-----------------------|-----------|---------|-----------|
| Nós expandidos        | 85.698    | 672     | **99,2%** |
| Tamanho da solução    | 22 passos | 22 passos | — (mesmo) |
| Razão (BFS / A\*)     | —         | —       | **127×**  |

Ambos os algoritmos encontraram a **mesma solução ótima de 22 passos**, confirmando a corretude das implementações. A diferença está no *custo de busca*.

### Por que a diferença é tão grande?

O BFS explora estados em ordem crescente de profundidade, sem qualquer noção de "direção". Para encontrar uma solução a 22 passos, ele precisa expandir **todos** os estados até profundidade 21 antes de finalmente encontrar o objetivo. No pior caso isso equivale a visitar dezenas de milhares de nós.

O A\*, por outro lado, usa a distância de Manhattan como bússola. A heurística admissível garante que ele nunca "desperdice" uma expansão num caminho que já é sabidamente pior do que o melhor candidato atual. Em vez de explorar em leque, ele avança preferencialmente em direção ao objetivo, podando ramificações inteiras do espaço de busca.

### Conclusão

A heurística de Manhattan transforma o A\* numa ferramenta drasticamente superior ao BFS para este problema. A vantagem cresce exponencialmente com a dificuldade do estado inicial: para estados com 20+ movimentos até o objetivo, o BFS se torna praticamente inviável (em tempo e memória), enquanto o A\* resolve em milissegundos.

Este experimento ilustra um princípio fundamental da IA: **informação heurística de qualidade vale mais do que poder computacional bruto**.